In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


In [ ]:
# 1. Define the exact file path
file_path = r'data/car_prices.csv'

In [ ]:
# 2. Load the raw data 
# header=0 tells pandas the first row contains column names
# low_memory=False helps with large files that have mixed data types
try:
    df = pd.read_csv(file_path, sep=',', header=0, low_memory=False)
    print("--- PHASE 0 COMPLETE ---")
    print(f"Successfully loaded {len(df):,} rows.")
    print(f"Columns found: {list(df.columns)}")
except FileNotFoundError:
    print(f"Error: Could not find the file at {file_path}. Please check the path.")
except Exception as e:
    print(f"An error occurred: {e}")

# Display the first few rows to verify the "Bulk Insert" worked
print("\nPreview of Raw Data:")
print(df.head(5))


In [ ]:
# PHASE 1: THE DATA ENGINEERING PIPELINE
# =============================================

# 1. INITIAL FILTERING (The SQL "WHERE" clause)
# We filter for valid years, prices, and non-null essential columns
df_clean = df[
    (df['state'].str.len() == 2) &
    (df['year'].between(1990, 2016)) &
    (df['sellingprice'] > 100) &
    (df['vin'].notnull()) &
    (df['make'].notnull()) &
    (df['model'].notnull()) &
    (df['vin'].str.len() >= 17)
].copy()

# 2. STANDARDIZED MAKE LOGIC (The first "CASE WHEN")
def clean_make(m):
    m = str(m).lower()
    mapping = {
        'dodge tk': 'Dodge', 'ford tk': 'Ford', 'ford truck': 'Ford',
        'chev truck': 'Chevrolet', 'gmc truck': 'GMC', 'mazda tk': 'Mazda',
        'hyundai tk': 'Hyundai', 'landrover': 'Land Rover', 'vw': 'Volkswagen',
        'mercedes': 'Mercedes-Benz', 'mercedes-b': 'Mercedes-Benz', 'dot': np.nan
    }
    return mapping.get(m, m.capitalize())

df_clean['Make'] = df_clean['make'].apply(clean_make)

# 3. STANDARDIZED BODY TYPES (The second "CASE WHEN")
# We use a map to group similar types into one category
body_map = {
    'SUV': 'SUV', 'SEDAN': 'Sedan', 'G SEDAN': 'Sedan', 'COUPE': 'Coupe', 
    'CONVERTIBLE': 'Convertible', 'HATCHBACK': 'Hatchback', 'WAGON': 'Wagon'
}
# Note: For brevity, I've mapped the majors; we use .get() to default to 'Other'
df_clean['Body Type'] = df_clean['body'].str.upper().map(body_map).fillna('Other')

# 4. TRANSMISSION LOGIC
df_clean['Transmission'] = 'Unknown'
df_clean.loc[df_clean['transmission'].str.contains('auto', case=False, na=False), 'Transmission'] = 'Automatic'
df_clean.loc[df_clean['transmission'].str.contains('man', case=False, na=False), 'Transmission'] = 'Manual'

# 5. CONDITION GRADE LOGIC
conditions = [
    (df_clean['condition'].isna()) | (df_clean['condition'] == 0),
    (df_clean['condition'] > 40),
    (df_clean['condition'] > 30),
    (df_clean['condition'] > 20),
    (df_clean['condition'] > 10)
]
choices = ['0 - Unknown', '5 - Excellent', '4 - Very Good', '3 - Good', '2 - Fair']
df_clean['Condition Grade'] = np.select(conditions, choices, default='1 - Poor')

# 6. CALCULATED COLUMNS (The SELECT logic)
df_clean['Profit/Loss'] = df_clean['sellingprice'] - df_clean['mmr']
df_clean['Vehicle Age'] = 2015 - df_clean['year']
# Substring for date: TRY_CAST(SUBSTRING(saledate, 5, 11) AS DATE)
df_clean['Date of Sale'] = pd.to_datetime(df_clean['saledate'].str[4:15], errors='coerce')

# 7. FINAL TABLE SELECTION (Analogous to SELECT INTO)
final_cols = {
    'year': 'Model Year', 'Make': 'Make', 'model': 'Model', 'trim': 'Trim',
    'Body Type': 'Body Type', 'vin': 'VIN', 'Transmission': 'Transmission',
    'state': 'State Code', 'condition': 'Raw Condition', 'Condition Grade': 'Condition Grade',
    'odometer': 'Mileage', 'sellingprice': 'Sale Price', 'mmr': 'Estimated Value',
    'Profit/Loss': 'Profit/Loss', 'Date of Sale': 'Date of Sale', 'Vehicle Age': 'Vehicle Age'
}

# Rename and keep only the columns we want
df_final = df_clean.rename(columns=final_cols)[list(final_cols.values())]

print("--- PHASE 1 COMPLETE ---")
print(f"Cleaned data contains {len(df_final):,} rows.")
print(df_final.head())




In [ ]:
df_final.tail()

In [ ]:
df_final.describe()

In [ ]:
df_final.info()

In [ ]:
df_final.duplicated().sum()

In [ ]:
df_final.isnull().sum()


In [ ]:
# Set visual style 
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


In [ ]:
# PHASE 2: EXPLORATORY DATA ANALYSIS (EDA)
# =============================================
print("\n--- EXECUTIVE KPI SUMMARY ---")
kpis = {
    "Total Revenue": f"${df_final['Sale Price'].sum():,.2f}",
    "Average Sale Price": f"${df_final['Sale Price'].mean():,.2f}",
    "Average Mileage": f"{df_final['Mileage'].mean():,.0f} miles",
    "Market Profit/Loss vs MMR": f"${df_final['Profit/Loss'].mean():,.2f}"
}
for k, v in kpis.items():
    print(f"{k}: {v}")


In [ ]:
# PHASE 3: BUSINESS SOLUTION VISUALIZATIONS
# =============================================

# =============================================
# 1. MARKET LIQUIDITY: Top 10 Makes by Volume
# Logic: Shows which brands dominate the auction floor.
# =============================================
plt.figure(figsize=(14, 7))
top_makes = df_final['Make'].value_counts().head(10).reset_index()
top_makes.columns = ['Make', 'Units Sold']

ax1 = sns.barplot(data=top_makes, x='Units Sold', y='Make', palette='magma')

# Add "Data Labels" (the actual numbers on the bars)
for i in ax1.containers:
    ax1.bar_label(i, padding=5, fmt='{:,.0f}', fontsize=10, fontweight='bold')

plt.title('Market Inventory Concentration: Top 10 High-Volume Brands', pad=20)
plt.xlabel('Total Units Sold (Volume)')
plt.ylabel('Vehicle Manufacturer')
plt.xlim(0, top_makes['Units Sold'].max() * 1.15) # Add space for labels
plt.tight_layout()
plt.savefig('1_market_liquidity.png', dpi=300)
plt.show()


In [ ]:
# 2. PRICING STRATEGY: Market Segmentation (Budget vs Luxury)
# Identifies the "Heart of the Market" and luxury outliers.
# =============================================
plt.figure(figsize=(14, 7))
median_price = df_final['Sale Price'].median()
avg_price = df_final['Sale Price'].mean()

# Filtering for visibility (removing ultra-rare $100k+ outliers for better distribution view)
sns.histplot(df_final[df_final['Sale Price'] < 70000]['Sale Price'], bins=60, kde=True, color='teal', alpha=0.6)

# Add Executive Reference Lines
plt.axvline(median_price, color='red', linestyle='--', lw=2, label=f'Median Price: ${median_price:,.0f}')
plt.axvline(avg_price, color='blue', linestyle=':', lw=2, label=f'Average Price: ${avg_price:,.0f}')

# Highlight the "Budget" vs "Mid-Range" split
plt.axvspan(0, 10000, color='gray', alpha=0.1, label='Budget Segment (<$10k)')

plt.title('Vehicle Price Distribution & Market Segmentation', pad=20)
plt.xlabel('Final Sale Price ($)')
plt.ylabel('Frequency (Number of Sales)')
plt.legend(frameon=True, facecolor='white')
plt.xticks(np.arange(0, 75000, 5000))
plt.tight_layout()
plt.savefig('2_price_distribution.png', dpi=300)
plt.show()




In [ ]:
# 3. ASSET DEPRECIATION: Mileage Brackets vs. Sale Price
# We group mileage into 10,000-mile "buckets" to see the average price drop.
plt.figure(figsize=(12, 6))

# Create 10k mileage bins
df_final['Mileage_Bin'] = (df_final['Mileage'] // 10000) * 10000

# Calculate Average Price per Bin
depreciation_data = df_final.groupby('Mileage_Bin')['Sale Price'].mean().reset_index()

# Filter for realistic mileage (e.g., up to 200,000 miles) to remove extreme outliers
depreciation_data = depreciation_data[depreciation_data['Mileage_Bin'] <= 200000]

# Plotting the Line & Area
sns.lineplot(data=depreciation_data, x='Mileage_Bin', y='Sale Price', color='red', lw=3, marker='o')
plt.fill_between(depreciation_data['Mileage_Bin'], depreciation_data['Sale Price'], color='red', alpha=0.1)

# Formatting for Business Readability
plt.title('Asset Depreciation: Average Realized Value by Mileage Milestone', fontsize=15)
plt.xlabel('Odometer Reading (Miles)', fontsize=12)
plt.ylabel('Average Sale Price ($)', fontsize=12)

# Add a "100k Mile" reference line (Psychological Barrier)
plt.axvline(100000, color='black', linestyle='--', alpha=0.6)
plt.text(105000, depreciation_data['Sale Price'].max()*0.8, 'The 100k Mile Threshold', fontsize=10, fontweight='bold')

plt.xticks(np.arange(0, 210000, 20000)) # Set clear x-axis intervals
plt.tight_layout()
plt.savefig('depreciation_milestones.png')
plt.show()



In [ ]:
# 4. REGIONAL PERFORMANCE: Top 10 States by Profit/Loss
# Identifies where MMR estimates are most accurate (or inaccurate).
# =============================================
plt.figure(figsize=(14, 7))
top_10_states = df_final['State Code'].value_counts().head(10).index
state_data = df_final[df_final['State Code'].isin(top_10_states)]

# Boxplot showing distribution of Profit/Loss per state
ax4 = sns.boxplot(data=state_data, x='State Code', y='Profit/Loss', palette='coolwarm', showfliers=False, notch=True)

# Add the "Zero Profit" baseline
plt.axhline(0, color='black', linestyle='-', lw=1.5, alpha=0.5)

plt.title('Regional Price Accuracy: Profit/Loss Variance vs. Market Estimates (MMR)', pad=20)
plt.xlabel('State (Top 10 by Volume)')
plt.ylabel('Variance from Estimated Value ($)')
plt.text(-0.4, state_data['Profit/Loss'].max()*0.05, 'Sold ABOVE Estimate', fontsize=10, color='green', fontweight='bold')
plt.text(-0.4, state_data['Profit/Loss'].min()*0.05, 'Sold BELOW Estimate', fontsize=10, color='red', fontweight='bold')

plt.tight_layout()
plt.savefig('4_regional_profitability.png', dpi=300)
plt.show()


In [ ]:
# 5. SALES VELOCITY: 7-Day Rolling Momentum
# Filters daily noise to see the true market trend.
# =============================================
plt.figure(figsize=(14, 7))
daily_sales = df_final.groupby('Date of Sale').size().rename('Daily Volume')
rolling_sales = daily_sales.rolling(window=7).mean()

plt.plot(daily_sales.index, daily_sales.values, color='lightgray', alpha=0.5, label='Raw Daily Volume')
plt.plot(rolling_sales.index, rolling_sales.values, color='blue', lw=3, label='7-Day Moving Average (Market Trend)')

# Annotate Peak Volume
peak_date = rolling_sales.idxmax()
peak_val = rolling_sales.max()
plt.annotate(f'Market Peak: {peak_val:,.0f} units', 
             xy=(peak_date, peak_val), xytext=(peak_date, peak_val + 500),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1),
             fontsize=10, fontweight='bold')

plt.title('Market Sales Velocity: Identifying Volume Trends & Seasonality', pad=20)
plt.xlabel('Transaction Date')
plt.ylabel('Number of Vehicles Sold')
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.savefig('5_sales_velocity.png', dpi=300)
plt.show()

print("\n--- ALL BUSINESS VISUALIZATIONS GENERATED SUCCESSFULLY ---")



In [ ]:
# PHASE 4: STRATEGIC BUSINESS DEEP DIVES
# =============================================
print("\n--- STARTING PHASE 4: STRATEGIC INSIGHTS ---")
# ---------------------------------------------------------
# 1. INVENTORY MIX: Which Body Types drive the most Revenue?
# Logic: Revenue vs. Volume. Some types might sell less but make more money.
# ---------------------------------------------------------
body_type_stats = df_final.groupby('Body Type').agg({
    'Sale Price': ['count', 'sum', 'mean']
}).reset_index()
body_type_stats.columns = ['Body Type', 'Units Sold', 'Total Revenue', 'Avg Price']
body_type_stats = body_type_stats.sort_values(by='Total Revenue', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=body_type_stats, x='Total Revenue', y='Body Type', palette='viridis')
plt.title('Revenue Contribution by Vehicle Body Type', fontsize=15)
plt.xlabel('Total Revenue ($)')
plt.savefig('6_body_type_revenue.png')



In [ ]:
# 2. SEASONALITY: Monthly Sales Trends
# Logic: Does the time of year affect how many cars we sell?
# ---------------------------------------------------------
df_final['Month'] = df_final['Date of Sale'].dt.month_name()
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']

monthly_sales = df_final.groupby('Month').size().reindex(month_order).dropna()

plt.figure(figsize=(12, 6))
monthly_sales.plot(kind='line', marker='s', color='darkblue', linewidth=3, markersize=10)
plt.fill_between(range(len(monthly_sales)), monthly_sales.values, color='skyblue', alpha=0.3)
plt.title('Auction Seasonality: Monthly Transaction Volume', fontsize=15)
plt.ylabel('Number of Vehicles Sold')
plt.xticks(range(len(monthly_sales)), monthly_sales.index)
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.savefig('7_monthly_trends.png')


In [ ]:
# 3. PROFITABILITY BY SELLER: The "Top 5 Big Players"
# Logic: Identifying our most important clients (Sellers).
# ---------------------------------------------------------
# Note: We need to pull 'seller' from df_clean since it wasn't in final_cols
df_final['Seller'] = df_clean['seller']
top_sellers = df_final.groupby('Seller')['Sale Price'].sum().nlargest(5).reset_index()

print("\n--- TOP 5 SELLERS BY REVENUE ---")
for i, row in top_sellers.iterrows():
    print(f"{i+1}. {row['Seller']}: ${row['Sale Price']:,.2f}")

# ---------------------------------------------------------
# 4. DATA-DRIVEN SUMMARY TABLE (The "Final Report")
# Logic: A consolidated view of our market segments.
# ---------------------------------------------------------
summary_table = df_final.groupby('Condition Grade').agg({
    'Sale Price': 'mean',
    'Mileage': 'mean',
    'Profit/Loss': 'mean'
}).round(2)

print("\n--- MARKET SEGMENT SUMMARY (BY CONDITION) ---")
print(summary_table)

print("\n--- PHASE 4 COMPLETE: STRATEGIC INSIGHTS GENERATED ---")
